# Регистрация baseline-модели в MLflow

In [1]:
import os
import json
import joblib
import pandas as pd
import mlflow

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
)

from dotenv import load_dotenv

/home/mle-user/projects/mle-mlflow/.venv/lib/python3.10/site-packages/pydantic/_internal/_config.py:408: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


## Подключение к MLflow и настройка S3

In [2]:
dotenv_path = os.path.expanduser("~/projects/mle-mlflow/.env")
load_dotenv(dotenv_path, override=True)

os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.getenv("MLFLOW_S3_ENDPOINT_URL", "https://storage.yandexcloud.net")
os.environ["S3_BUCKET_NAME"] = os.getenv("S3_BUCKET_NAME")
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

## Загрузка модели и данных предыдущего этапа

In [3]:
EXPERIMENT_NAME = "project_model_improvement"
RUN_NAME = "baseline_registration"
REGISTRY_MODEL_NAME = "project_churn_model"

MODEL_PATH = os.path.expanduser("~/projects/mle-dvc/models/fitted_model.pkl")
DATA_PATH = os.path.expanduser("~/projects/mle-dvc/data/initial_data.csv")
CV_RES_PATH = os.path.expanduser("~/projects/mle-dvc/cv_results/cv_res.json")

pipeline = joblib.load(MODEL_PATH)
data = pd.read_csv(DATA_PATH, index_col="customer_id")

with open(CV_RES_PATH) as f:
    cv_metrics = json.load(f)

print(cv_metrics)
print(data.shape)

{'fit_time': 7.386, 'score_time': 0.128, 'test_f1': 1.0, 'test_roc_auc': 1.0}
(7001, 21)


## Оценка baseline-модели на тестовой выборке

In [4]:
X = data.drop(columns=["target"])
y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

prediction = pipeline.predict(X_test)
proba = pipeline.predict_proba(X_test)[:, 1]

metrics = {}
_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize='all').ravel()
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = roc_auc_score(y_test, proba)
metrics["precision"] = precision_score(y_test, prediction)
metrics["recall"] = recall_score(y_test, prediction)
metrics["f1"] = f1_score(y_test, prediction)
metrics["logloss"] = log_loss(y_test, prediction)

print(metrics)

{'err1': 0.0, 'err2': 0.2633832976445396, 'auc': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'logloss': 2.2204460492503136e-16}


## Логирование и регистрация модели в MLflow

In [5]:
params = {
    "auto_class_weights": "Balanced",
    "one_hot_drop": "if_binary",
    "n_splits": 5,
}

# пропуски в объектных колонках (end_date) мешают построению сигнатуры,
# поэтому для неё делаем копию с заполнением
signature = mlflow.models.infer_signature(X_test.fillna(""), prediction)
input_example = X_test[:10].fillna("")

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)

    # метрики кросс-валидации из предыдущего спринта
    cv_metrics_clean = {f"cv_{k.replace('test_', '')}": v for k, v in cv_metrics.items()}
    mlflow.log_metrics(cv_metrics_clean)

    mlflow.log_artifact(CV_RES_PATH, "cv_results")

    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="models",
        registered_model_name=REGISTRY_MODEL_NAME,
        signature=signature,
        input_example=input_example,
        pip_requirements=os.path.expanduser("~/projects/mle-mlflow/requirements.txt"),
        await_registration_for=60,
    )

print(run_id)

/home/mle-user/projects/mle-mlflow/.venv/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None
Successfully registered model 'project_churn_model'.
2026/11/24 04:01:29 INFO mlflow.tracking._model_registry.client: Waiting up to 60 seconds for model ve

5d780e1504ad49a5b973b62e77e68ca4


Created version '1' of model 'project_churn_model'.


## Проверка реестра моделей

In [6]:
client = mlflow.MlflowClient()

model_versions = client.search_model_versions(filter_string=f"name = '{REGISTRY_MODEL_NAME}'")
for mv in model_versions:
    print(f"version {mv.version}: {mv.current_stage}")

version 1: None
